# 4.8 多一層，是多了什麼計算與成本？


一層 block 讀前文並修正表示；第二層拿的是**第一層更新後的表示**，再讀取與加工。層數叫**深度**，不是把相同原卡平均三次。更多層有機會構成更複雜規則，但每層也要儲存自己的配方並增加運算。

先固定二十候選、每位置八格、位置表最多 128 格，只改深度 1、2、3。希望檢查輸出仍是相同形狀，並數清每多一層加多少參數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.model import TinyLM, ModelConfig

for depth in [1, 2, 3]:
    model = TinyLM(ModelConfig(vocab_size=20, width=8, layers=depth))
    count = model.description()["parameters"]
    logits = model(torch.tensor([[1, 2]]))["logits"]
    print("層數", depth, "參數格數", count, "輸出", logits.shape)
    assert logits.shape == (1, 2, 20)

三次輸出都為 `[1,2,20]`：一筆兩位置，每位置二十候選。中間寬度八格不會把候選數變成八。參數格數為：

| 深度 | 總參數格數 | 比前一個多 |
| --- | ---: | ---: |
| 1 | 2,200 | — |
| 2 | 3,040 | 840 |
| 3 | 3,880 | 840 |

為什麼三層不是一層的三倍？字表、位置表、最後 LN 與輸出層只在入口出口各存一套，本例共 1,360 格；各 block 則獨立儲存 840 格。因此總數為 `1360+深度×840`。

更深也需要更多前向、求導與中間材料。寬度增加則影響許多矩陣的兩邊：8×8 變 16×16 是四倍；字表與位置表只有一邊是寬度，則約兩倍。不能將層數、寬度、候選數都用同一倍率估算。

這幾份模型都未訓練，只檢查結構，不提供三層更準的證據。要比較品質，應固定資料，交代更新步數或計算預算，並在保留資料上評估。更多參數也可能只記住少量訓練句。

練習只把寬度從 8 改成 16，候選仍二十，輸出保持 `[1,2,20]`。預測參數增加但非全部翻倍，再執行記錄每多一層的差額，與原本 840 比較。

<details>
<summary>補充與重做</summary>

本例入口出口：字表 `20×8=160`，位置表 `128×8=1024`，最後 LN 的倍率與偏移 `8+8=16`，輸出權重 `8×20=160`，共 1360。每個 block 的明細如下。

| 每 block 的可調數字 | 演演演算法 | 格數 |
| --- | --- | ---: |
| Q、K、V 與注意力輸出的四張權重 | `4×8×8` | 256 |
| FFN 擴張、縮回權重 | `32×8+8×32` | 512 |
| FFN 兩份偏移 | `32+8` | 40 |
| 兩個 LN 的倍率與偏移 | `2×(8+8)` | 32 |

合計 840。注意力及候選輸出在這個設定沒有偏移；FFN 兩層有偏移，非線性本身無可調數字。

現有 [T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4) 的另一個模型字表 264、寬度 64、位置上限 128、兩層，共 `42112+2×49728=141568` 格。264 是文字內容與控制編號的完整候選表，不是 264 箇中文字。那份記錄示範訓練、存檔與生成；沒有另做同預算一層對照，不能用來證明兩層更好。完整操作保留在 T.4。

</details>
